# WarnaIn v3 — Training & Benchmark 3 Paradigma Deep Learning
### Komparasi Arsitektur Restorasi Citra Monokrom CIE $L^*a^*b^*$ (LO3 & LO4)

Notebook ini dirancang agar dapat dijalankan secara **lokal (VS Code / Jupyter)** maupun di **Google Colab / Kaggle GPU** untuk melatih dan mengevaluasi tiga keluarga model dengan paradigma berbeda:

1. **Model 1 — CNN / Autoencoder-based (`ResNetUNetColorizer`)**:
   * **Arsitektur**: ResNet-18 Backbone (ImageNet pre-trained, adaptasi `conv1` 1-channel) + U-Net Progressive Transposed Convolution Decoder.
   * **Fungsi Objektif**: Regresi piksel langsung via kombinasi $\mathcal{L}_{\text{SmoothL1}} + \mathcal{L}_{\text{MSE}}$.
2. **Model 2 — Adversarial / GAN-based (`Pix2PixColorizer`)**:
   * **Arsitektur**: Conditional GAN dengan U-Net Generator + **PatchGAN 70×70 Discriminator** yang menerima pasangan input `[L*, a*b*]` (3 channel).
   * **Fungsi Objektif**: $\mathcal{L}_{\text{cGAN}}(G, D) + \lambda \mathcal{L}_{L1}(G)$ dengan $\lambda = 100$.
3. **Model 3 — Transformer-based (`TransformerColorizer`)**:
   * **Arsitektur**: Swin-UNet / Restormer hierarchical block dengan **Windowed Multi-Head Self-Attention (W-MSA & SW-MSA)**.
   * **Fungsi Objektif**: Optimasi `AdamW` dengan $\mathcal{L}_{\text{SmoothL1}}$ untuk koherensi semantik global.

---
**Kontrak Input-Output Standar:**
* **Input**: Tensor Luminansi $L^*$ berdimensi `(B, 1, 224, 224)` dalam rentang `[-1, 1]`.
* **Output**: Tensor Prediksi Krominansi $a^*b^*$ berdimensi `(B, 2, 224, 224)` dalam rentang `[-1, 1]` (aktivasi akhir `Tanh`).

In [1]:
import os
import sys
import time
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Deteksi path root WarnaIn3 secara otomatis
CWD = os.path.abspath(os.getcwd())
if os.path.basename(CWD) == "notebooks":
    V3_ROOT = os.path.abspath(os.path.join(CWD, ".."))
elif os.path.isdir(os.path.join(CWD, "WarnaIn3")):
    V3_ROOT = os.path.abspath(os.path.join(CWD, "WarnaIn3"))
else:
    V3_ROOT = CWD

if V3_ROOT not in sys.path:
    sys.path.insert(0, V3_ROOT)

from src.model import (
    ResNetUNetColorizer,
    Pix2PixColorizer,
    TransformerColorizer,
    build_colorizer,
)
from src.dataset import ColorizationDataset, resolve_data_path
from src.utils import (
    rgb_to_lab,
    lab_to_rgb,
    tensor_to_rgb,
    image_to_tensor,
    calculate_psnr,
    calculate_ssim,
    calculate_mae,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR = os.path.join(V3_ROOT, "checkpoints")
REPORTS_DIR = os.path.join(V3_ROOT, "reports")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"[INFO] WarnaIn3 Root : {V3_ROOT}")
print(f"[INFO] Active Device : {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

d:\Python\environments\deep_learning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'torch'

In [ ]:
%pip install torch

## 1. Persiapan Dataset & DataLoader CIE $L^*a^*b^*$
Dataset otomatis membaca split `data/splits/train`, `data/splits/val`, dan `data/splits/test` dari workspace utama tanpa perlu duplikasi file.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
MAX_TRAIN_SAMPLES = 400  # Ubah ke None jika ingin melatih pada seluruh data train (4000 citra)
MAX_VAL_SAMPLES = 100

train_dataset = ColorizationDataset(
    image_dir="data/splits/train",
    image_size=IMG_SIZE,
    is_training=True,
    max_samples=MAX_TRAIN_SAMPLES,
)
val_dataset = ColorizationDataset(
    image_dir="data/splits/val",
    image_size=IMG_SIZE,
    is_training=False,
    max_samples=MAX_VAL_SAMPLES,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

L_sample, ab_sample = next(iter(train_loader))
print(f"Train samples : {len(train_dataset)} | Val samples : {len(val_dataset)}")
print(f"Input L* batch shape  : {L_sample.shape} | range: [{L_sample.min():.2f}, {L_sample.max():.2f}]")
print(f"Target ab batch shape : {ab_sample.shape} | range: [{ab_sample.min():.2f}, {ab_sample.max():.2f}]")

## 2. Verifikasi Arsitektur & Jumlah Parameter Ketiga Model
Memastikan ketiga paradigma memenuhi kontrak dimensi `(B, 1, 224, 224) -> (B, 2, 224, 224)` dalam rentang `[-1, 1]`.

In [ ]:
model_cnn = ResNetUNetColorizer(pretrained_backbone=True, use_pretrained_fallback=False).to(DEVICE)
model_gan = Pix2PixColorizer(use_pretrained_fallback=False).to(DEVICE)
model_vit = TransformerColorizer(embed_dim=48, window_size=7, use_pretrained_fallback=False).to(DEVICE)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad) / 1e6

with torch.no_grad():
    dummy_L = torch.randn(2, 1, 224, 224, device=DEVICE).clamp(-1, 1)
    out_cnn = model_cnn(dummy_L)
    out_gan = model_gan(dummy_L)
    disc_patch = model_gan.discriminate(dummy_L, out_gan)
    out_vit = model_vit(dummy_L)

print(f"1. ResNetUNetColorizer   | Params: {count_params(model_cnn):.2f}M | Output: {tuple(out_cnn.shape)}")
print(f"2. Pix2PixColorizer (G+D)| Params: {count_params(model_gan):.2f}M | Gen Out: {tuple(out_gan.shape)} | PatchGAN Out: {tuple(disc_patch.shape)}")
print(f"3. TransformerColorizer  | Params: {count_params(model_vit):.2f}M | Output: {tuple(out_vit.shape)}")

## 3. Pipeline Training Ketiga Paradigma

### 3.1 Training Model 1: CNN (`ResNetUNetColorizer` — Smooth L1 + MSE Regression)

In [ ]:
def train_cnn_resnet_unet(model, train_loader, val_loader, epochs=5, lr=2e-4):
    optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999))
    l1_crit = nn.SmoothL1Loss()
    mse_crit = nn.MSELoss()
    history = {"train_loss": [], "val_loss": []}
    best_val = float("inf")

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for L_batch, ab_batch in tqdm(train_loader, desc=f"[CNN ResNet-UNet] Epoch {epoch}/{epochs}", leave=False):
            L_batch, ab_batch = L_batch.to(DEVICE), ab_batch.to(DEVICE)
            optimizer.zero_grad()
            pred_ab = model(L_batch)
            loss = 0.7 * l1_crit(pred_ab, ab_batch) + 0.3 * mse_crit(pred_ab, ab_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * L_batch.size(0)

        train_loss = running_loss / max(1, len(train_loader.dataset))
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for L_batch, ab_batch in val_loader:
                L_batch, ab_batch = L_batch.to(DEVICE), ab_batch.to(DEVICE)
                pred_ab = model(L_batch)
                loss = 0.7 * l1_crit(pred_ab, ab_batch) + 0.3 * mse_crit(pred_ab, ab_batch)
                val_loss += loss.item() * L_batch.size(0)
        val_loss /= max(1, len(val_loader.dataset))

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"Epoch {epoch:02d}/{epochs:02d} — Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            save_path = os.path.join(CKPT_DIR, "resnet_unet_best.pth")
            torch.save({"model_state_dict": model.state_dict(), "arch": "resnet_unet", "val_loss": best_val}, save_path)
    return history

# Ubah EPOCHS sesuai kebutuhan (misal 10-30 epoch di GPU Colab)
EPOCHS_CNN = 5
# history_cnn = train_cnn_resnet_unet(model_cnn, train_loader, val_loader, epochs=EPOCHS_CNN)

### 3.2 Training Model 2: Adversarial cGAN (`Pix2PixColorizer` — U-Net Generator + PatchGAN 70×70)

In [ ]:
def train_pix2pix_gan(model: Pix2PixColorizer, train_loader, epochs=5, lr=2e-4, lambda_l1=100.0):
    opt_G = optim.Adam(model.generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = optim.Adam(model.discriminator.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion_GAN = nn.BCEWithLogitsLoss()
    criterion_L1 = nn.L1Loss()
    history = {"G_loss": [], "D_loss": []}

    for epoch in range(1, epochs + 1):
        model.train()
        g_running, d_running = 0.0, 0.0

        for L_batch, ab_batch in tqdm(train_loader, desc=f"[GAN Pix2Pix] Epoch {epoch}/{epochs}", leave=False):
            L_batch, ab_batch = L_batch.to(DEVICE), ab_batch.to(DEVICE)

            # ------------------------------------------
            # 1. Update PatchGAN Discriminator: max log(D(L, ab)) + log(1 - D(L, G(L)))
            # ------------------------------------------
            opt_D.zero_grad()
            fake_ab = model.generator(L_batch)

            pred_real = model.discriminate(L_batch, ab_batch)
            real_labels = torch.ones_like(pred_real, device=DEVICE)
            loss_D_real = criterion_GAN(pred_real, real_labels)

            pred_fake = model.discriminate(L_batch, fake_ab.detach())
            fake_labels = torch.zeros_like(pred_fake, device=DEVICE)
            loss_D_fake = criterion_GAN(pred_fake, fake_labels)

            loss_D = 0.5 * (loss_D_real + loss_D_fake)
            loss_D.backward()
            opt_D.step()

            # ------------------------------------------
            # 2. Update U-Net Generator: min log(1 - D(L, G(L))) + lambda * L1(G(L), ab)
            # ------------------------------------------
            opt_G.zero_grad()
            pred_fake_for_G = model.discriminate(L_batch, fake_ab)
            loss_G_GAN = criterion_GAN(pred_fake_for_G, real_labels)
            loss_G_L1 = criterion_L1(fake_ab, ab_batch) * lambda_l1
            loss_G = loss_G_GAN + loss_G_L1
            loss_G.backward()
            opt_G.step()

            g_running += loss_G.item() * L_batch.size(0)
            d_running += loss_D.item() * L_batch.size(0)

        g_epoch = g_running / max(1, len(train_loader.dataset))
        d_epoch = d_running / max(1, len(train_loader.dataset))
        history["G_loss"].append(g_epoch)
        history["D_loss"].append(d_epoch)
        print(f"Epoch {epoch:02d}/{epochs:02d} — Generator Loss: {g_epoch:.4f} | PatchGAN Disc Loss: {d_epoch:.4f}")

        save_path = os.path.join(CKPT_DIR, "pix2pix_best.pth")
        torch.save({"model_state_dict": model.state_dict(), "arch": "pix2pix"}, save_path)
    return history

EPOCHS_GAN = 5
# history_gan = train_pix2pix_gan(model_gan, train_loader, epochs=EPOCHS_GAN)

### 3.3 Training Model 3: Vision Transformer (`TransformerColorizer` — Swin-UNet Self-Attention)

In [ ]:
def train_transformer_swin(model: TransformerColorizer, train_loader, val_loader, epochs=5, lr=1.5e-4):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.SmoothL1Loss()
    history = {"train_loss": [], "val_loss": []}
    best_val = float("inf")

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for L_batch, ab_batch in tqdm(train_loader, desc=f"[Transformer Swin-UNet] Epoch {epoch}/{epochs}", leave=False):
            L_batch, ab_batch = L_batch.to(DEVICE), ab_batch.to(DEVICE)
            optimizer.zero_grad()
            pred_ab = model(L_batch)
            loss = criterion(pred_ab, ab_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * L_batch.size(0)
        scheduler.step()

        train_loss = running_loss / max(1, len(train_loader.dataset))
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for L_batch, ab_batch in val_loader:
                L_batch, ab_batch = L_batch.to(DEVICE), ab_batch.to(DEVICE)
                pred_ab = model(L_batch)
                val_loss += criterion(pred_ab, ab_batch).item() * L_batch.size(0)
        val_loss /= max(1, len(val_loader.dataset))

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"Epoch {epoch:02d}/{epochs:02d} — Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            save_path = os.path.join(CKPT_DIR, "transformer_best.pth")
            torch.save({"model_state_dict": model.state_dict(), "arch": "transformer", "val_loss": best_val}, save_path)
    return history

EPOCHS_VIT = 5
# history_vit = train_transformer_swin(model_vit, train_loader, val_loader, epochs=EPOCHS_VIT)

## 4. Visualisasi Komparasi Head-to-Head 3 Paradigma (Siap Masuk Bab 4 Laporan)
Sel di bawah ini memanggil `build_colorizer()` untuk ketiga model dan menampilkan perbandingan visual berdampingan beserta nilai **PSNR / SSIM / MAE**.

In [ ]:
m_cnn_eval = build_colorizer("resnet_unet", device=str(DEVICE))
m_gan_eval = build_colorizer("pix2pix", device=str(DEVICE))
m_vit_eval = build_colorizer("transformer", device=str(DEVICE))

fig, axes = plt.subplots(1, 5, figsize=(18, 4.2), dpi=120)
L_single = L_sample[0:1].to(DEVICE)
ab_gt = ab_sample[0:1].to(DEVICE)

with torch.no_grad():
    ab_cnn = m_cnn_eval(L_single)
    ab_gan = m_gan_eval(L_single)
    ab_vit = m_vit_eval(L_single)

gt_rgb = tensor_to_rgb(L_single, ab_gt)
gray_rgb = tensor_to_rgb(L_single, torch.zeros_like(ab_gt))
cnn_rgb = tensor_to_rgb(L_single, ab_cnn)
gan_rgb = tensor_to_rgb(L_single, ab_gan)
vit_rgb = tensor_to_rgb(L_single, ab_vit)

plates = [
    ("1. Input Grayscale (L*)", gray_rgb),
    (f"2. CNN ResNet-UNet\nPSNR: {calculate_psnr(gt_rgb, cnn_rgb):.1f}dB | SSIM: {calculate_ssim(gt_rgb, cnn_rgb):.3f}", cnn_rgb),
    (f"3. GAN Pix2Pix\nPSNR: {calculate_psnr(gt_rgb, gan_rgb):.1f}dB | SSIM: {calculate_ssim(gt_rgb, gan_rgb):.3f}", gan_rgb),
    (f"4. Swin-UNet ViT\nPSNR: {calculate_psnr(gt_rgb, vit_rgb):.1f}dB | SSIM: {calculate_ssim(gt_rgb, vit_rgb):.3f}", vit_rgb),
    ("5. Ground Truth RGB", gt_rgb),
]

for ax, (title, img) in zip(axes, plates):
    ax.imshow(img)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Eksekusi Benchmark Kuantitatif Otomatis pada Test Set COCO 2017
Menjalankan evaluasi ketiga paradigma dan menyimpan hasilnya ke `reports/benchmark_summary.csv` dan `reports/benchmark_table_bab4.md`.

In [ ]:
from src.evaluate import benchmark_three_paradigms

# Ubah max_samples=500 untuk evaluasi penuh 500 citra COCO Test Set
summary_rows = benchmark_three_paradigms(
    test_dir="data/splits/test",
    img_size=224,
    device=str(DEVICE),
    max_samples=500,
)

df_benchmark = pd.DataFrame(summary_rows)
df_benchmark